In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Sem6_ai/week8/trum_tweet_sentiment_analysis.csv")

## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [7]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

stop_words = set(stopwords.words("english"))

def lower_order(text):
    return str(text).lower()

def remove_urls(text):
    return re.sub(r"http\S+|www\.\S+", " ", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r" ", text)

def removeunwanted_characters(text):
    text = re.sub(r"@[A-Za-z0-9_]+"," ", text)
    text = re.sub(r"#[A-Za-z0-9_]+"," ", text)
    text = re.sub(r"[^0-9A-Za-z ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    data = lower_order(dataset)
    data = remove_urls(data)
    data = remove_emoji(data)
    data = removeunwanted_characters(data)

    tokens = data.split()
    tokens = [word for word in tokens if word not in stop_words]

    wordnet = WordNetLemmatizer()
    porter = PorterStemmer()

    if rule == "lemmatize":
        tokens = [wordnet.lemmatize(word, pos='v') for word in tokens]
    elif rule == "stem":
        tokens = [porter.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [8]:
print(df.columns)

Index(['text', 'Sentiment'], dtype='object')


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df = pd.read_csv("/content/drive/MyDrive/Sem6_ai/week8/trum_tweet_sentiment_analysis.csv")

df["clean_text"] = df["text"].apply(text_cleaning_pipeline)

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["Sentiment"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94    248563
           1       0.89      0.85      0.87    121462

    accuracy                           0.92    370025
   macro avg       0.91      0.90      0.91    370025
weighted avg       0.92      0.92      0.92    370025

